# Mechanical mode comb: supplementary-information figures (archived)

Heterodyne spectra $S_{het}(\omega)$ of $N_p$ beads (x and y each) in the 'pairs' and 'equal
spacing' Ω models, stacked vertically as one parameter varies, for the supplementary
information of the mechanical-mode-comb work. The first cell defines `sample_with_rms`, which
draws random couplings with a prescribed RMS; the second cell makes the figures for $N_p$ = 2 and
4 with random couplings of RMS $g_{RMS}/2\pi$ = 10, 20, 40 and 60 kHz; the last cell varies a
uniform coupling for 10 beads.

**Runtime:** about a minute.

**Updated Nov 2026** to current levisim:

- the toy parameter `'gs'` (per-dimension couplings) → `'gjn_dict'`, its name since Dec 2025.
  Today's `MultiTrap` ignores the old key, so without the rename every coupling would silently be 0;
- `calculate_heterodyne_spectrum` → `heterodyne_spectrum`;
- the inline xmgrace-style rcParams → `set_plt_rcParams_to_xmgrace_style()` (the same settings);
- the random couplings are seeded (`np.random.seed(0)`), so they differ from the unseeded draws
  in the original run.

In [ ]:
import numpy as np

# seeded, so the random couplings (and hence the figures) are reproducible
np.random.seed(0)

def sample_with_rms(n, a, b, R, max_iter=1000, tol=1e-6):
    # Check if R is feasible
    R_min = max(0, min(abs(a), abs(b)) if a * b <= 0 else min(a, b))
    R_max = max(abs(a), abs(b))
    if R < R_min or R > R_max:
        raise ValueError(f"R = {R} is not achievable in the range [{a}, {b}]")

    # Generate initial random numbers in [a, b]
    x = np.random.uniform(a, b, n)

    for _ in range(max_iter):
        # Compute current RMS and rescale
        current_R = np.sqrt(np.mean(x**2))
        x = x * (R / current_R)

        # Clip values outside [a, b]
        x = np.clip(x, a, b)

        # Check if RMS is close enough
        if np.abs(np.sqrt(np.mean(x**2)) - R) < tol:
            return x

    raise RuntimeError("Failed to converge")

# Example usage:
n = 10       # Number of samples
a = 5      # Lower bound
b = 100       # Upper bound
R = 40     # Desired RMS

samples = sample_with_rms(n, a, b, R)
print("Samples:", samples)
print("Actual RMS:", np.sqrt(np.mean(samples**2)))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

# the same xmgrace-inspired style the original defined inline
set_plt_rcParams_to_xmgrace_style()
# curves touch the left and right edges of the axes
plt.rcParams['axes.xmargin'] = 0

def make_Ω_model_dict(Np, model_name):
    if model_name == 'pairs':
        if Np == 1:
            return {
                'Ω_model': model_name,
                'Ω_first': 250*kHz,
                'Ω_interpair_gap': 15*kHz,
                'Ω_intrapair_gap': 15*kHz,
            }
        if Np <= 5:
            return {
                'Ω_model': model_name,
                'Ω_first': 325*kHz,
                'Ω_interpair_gap': 15*kHz,
                'Ω_intrapair_gap': 1.5*kHz,
            }
        if Np <= 10:
            return {
                'Ω_model': model_name,
                'Ω_first': 350*kHz,
                'Ω_interpair_gap': 15*kHz,
                'Ω_intrapair_gap': 1.5*kHz,
            }
    if model_name == 'equal spacing':
        return {
            'Ω_model': model_name,
            'Ω_mid': 300*kHz,
            'Ω_span': Np * 10*kHz,
        }

def make_g_plots(
    # number of particles trapped
    Np,

    # uniform/equal spacing or pairs of x,y frequencies
    Ω_model_name,

    # 4 different parameters to vary per plot
    vary_ratio=False,
    vary_g=False,
    vary_Δ=False,
    vary_κ=False,
    custom_gs=False,

    **kwargs,
):

    gx = kwargs.get('gx', -40*kHz)
    def make_g_dict(gx=None, gy=None, Np=Np):
        g_dict = {}
        for ii in range(Np):
            g_dict[f'x{ii + 1}'] = gx
            g_dict[f'y{ii + 1}'] = gy
        return g_dict

    if vary_g:
        g_list = kwargs.get('g_list', -np.array([10, 20, 40, 80]) * kHz)
        gs_dicts = [make_g_dict(gx=g, gy=g) for g in g_list]
        descriptions = [f'g$/2π$ = {g / kHz:.0f} kHz' for g in g_list]

    elif vary_ratio:
        ratios = kwargs.get('ratios', [0.10, 0.25, 0.75, 1.00])
        gs_dicts = [make_g_dict(gx=gx, gy=ratio * gx) for ratio in ratios]
        descriptions = [f'g$_y$/g$_x$ = {ratio:.2f}' for ratio in ratios]

    elif vary_Δ:
        Δ_list = kwargs.get('Δ_list', -np.array([900, 450, 225, 100]) * kHz)
        descriptions = [f'Δ$/2π$ = {Δ / kHz:.0f} kHz' for Δ in Δ_list]

    elif vary_κ:
        κ_list = kwargs.get('κ_list', np.array([800, 400, 200, 100]) * kHz)
        descriptions = [f'κ$/2π$ = {κ / kHz:.0f} kHz' for κ in κ_list]

    elif custom_gs:
        toy_gs_list = kwargs.get('toy_gs_list')
        descriptions = kwargs.get('descriptions')


    # setup the subplot mosaic in xmgrace style
    num_plots = len(descriptions)
    fig = plt.figure(figsize=(4, 2 * num_plots))
    ax_dict = fig.subplot_mosaic(
        [[f'plot{ii}'] for ii in range(num_plots)],
        sharex=True,
        gridspec_kw={'hspace': 0},
    )

    Ω_model_dict = make_Ω_model_dict(Np=Np, model_name=Ω_model_name)
    low, high, low_res = kwargs.get('plot_range', (0, 400, 2.5))
    def get_hd_plot_resolution_params(Np):
        if Np == 1:
            return 200, 310, 0.01
        if Np <= 5:
            return 245, 330, 0.1
        if Np <= 10:
            return 205, 355, 0.05
    hd_low, hd_high, hd_res = get_hd_plot_resolution_params(Np=Np)
    ω_range = np.concatenate(
        (
            np.arange(low, hd_low, low_res) * kHz,
            np.arange(hd_low, hd_high, hd_res) * kHz,
            np.arange(hd_high, high, low_res) * kHz,
        ),
    )

    for ii, description in enumerate(descriptions):
        print(description)

        toy_gs = make_g_dict(gx=gx, gy=gx)
        if vary_ratio or vary_g:
            toy_gs = gs_dicts[ii]

        toy_params = {
            'gjn_dict': toy_gs,
            'use_Γ_formula': True,
        }

        if custom_gs:
            toy_params = {
                'gjn_dict': toy_gs_list[ii],
                'use_Γ_formula': True,
            }

        toy_params.update(Ω_model_dict)

        toy_Δ = -450*kHz if not vary_Δ else Δ_list[ii]
        toy_κ = 200*kHz if not vary_κ else κ_list[ii]

        cavity_kwargs = {
            'Δ': toy_Δ,
            'κ': toy_κ,
        }
        cavity_kwargs.update(kwargs.get('cavity_kwargs_override', {}))

        gas_kwargs = {
            'P': 1e-7*mbar,
            'T': 300*K,
            'vth': 500, # m/s
        }
        gas_kwargs.update(kwargs.get('gas_kwargs_override', {}))

        mtrap_Nx2 = MultiTrap(
            num_traps=Np,
            Cartesian_axes_per_trap=2,
            ω_range=ω_range,
            cavity_kwargs=cavity_kwargs,
            gas_kwargs=gas_kwargs,
            toy_params=toy_params,
        )
        plot_values = mtrap_Nx2.heterodyne_spectrum(QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)

        ax = ax_dict[f'plot{ii}']
        ax.fill_between(
            mtrap_Nx2.ω_range / kHz,
            plot_values,
            hatch='.....',           # Dot pattern (adjust '.' count for density)
            edgecolor='blue',        # Edge color of dots
            facecolor='none',        # No solid fill (only dots)
            linewidth=1.5,           # No edge line (only dots)
            alpha=0.8,               # Adjust dot opacity
        )
        ax.set_yscale('log')

        ax.set_yticks([0.1, 1, 10, 100])
        ax.get_yaxis().set_major_formatter(plt.FormatStrFormatter('%.1f'))
        ax.set_ylim(1e-1, 0.99999*1e2)
        ax.set_xlim(min(ω_range) / kHz, max(ω_range) / kHz)

        # --- Add transparent textbox to each subplot ---
        textbox = {
            's': description,      # Text content
            'fontsize': 15,         # Font size
            'fontname': 'Arial',    # Font family (to match Grace)
            'bbox': {
                'facecolor': 'white',  # Background color
                'alpha': 0.5,         # Transparency (0=invisible, 1=opaque)
                'edgecolor': 'none',   # No border
                'boxstyle': 'round,pad=0.2'  # Rounded corners (optional)
            },
            'ha': 'left',           # Horizontal alignment ('left'|'center'|'right')
            'va': 'top',            # Vertical alignment ('top'|'center'|'bottom')
            'zorder': 100,          # Ensure text is on top
        }

        # Add to subplot 1 (top-left corner)
        ax.text(
            0.15, 0.85,
            # axes-relative coordinates
            transform=ax.transAxes,
            **textbox,
        )

        # textbox.update({
        #         's': Ω_model_name,
        #         'fontsize': 12,
        #         'color': 'gray',
        #         'alpha': 0.5,
        # })
        # ax.text(
        #     0.65, 0.85,
        #     # axes-relative coordinates
        #     transform=ax.transAxes,
        #     **textbox,
        # )

    # fig.suptitle(f'N$_p$={Np}', fontsize=20)
    fig.suptitle(f'N$_p$={Np}; {Ω_model_name}', fontsize=20)
    ax_dict[f'plot{len(descriptions)-1}'].set_xlabel('Frequency (kHz)', fontsize=15)
    fig.text(-0.02, 0.5, 'Heterodyne spectra', rotation=90, va='center', fontsize=15)
    plt.tight_layout()
    plt.show()


for Np in [
    # 1,
    2,
    # 3,
    4,
    # 5,
    8,
    # 10,
]:
    n, a, b = Np * 2, 5, 100

    g_RMSs = [
        10,
        20,
        # 30,
        40,
        # 50,
        60,
        # 70,
        # 80,
    ]
    descriptions = [r'$g_{RMS} / 2π$' + f' = -{g_RMS} kHz' for g_RMS in g_RMSs]
    all_samples = {g_RMS: sample_with_rms(n, a, b, g_RMS) for g_RMS in g_RMSs}

    for samples in all_samples.values():
        print("Samples:", samples)
        print("Actual RMS:", np.sqrt(np.mean(samples**2)))

    def make_custom_g_dict(g_list, Np=Np):
        g_dict = {}
        iter_g_list = iter(g_list)
        for ii in range(Np):
            g_dict[f'x{ii + 1}'] = next(iter_g_list)
            g_dict[f'y{ii + 1}'] = next(iter_g_list)
        return g_dict

    toy_gs_list = [make_custom_g_dict(-np.array(samples) * kHz) for samples in all_samples.values()]

    make_g_plots(Np, 'pairs', custom_gs=True, descriptions=descriptions, toy_gs_list=toy_gs_list)
    make_g_plots(Np, 'equal spacing', custom_gs=True, descriptions=descriptions, toy_gs_list=toy_gs_list)

# for model in [
#     'pairs',
#     # 'equal spacing',
# ]:
#     for Np in [
#         1,
#         # 2,
#         # 4,
#         5,
#         # 8,
#         10,
#     ]:
#         # make_g_plots(Np, model, vary_g=True)
#         make_g_plots(Np, model, vary_ratio=True)
#         # make_g_plots(Np, model, vary_Δ=True)
#         # make_g_plots(Np, model, vary_κ=True)


#         # make_g_plots(
#         #     Np,
#         #     model,
#         #     vary_Δ=True,
#         #     Δ_list=-np.linspace(150, 600, 4) * kHz,
#         #     plot_range=(0, 600, 2.5),
#         # )






In [ ]:
# make_g_plots(10, 'pairs', vary_g=True, g_list=-np.array([35, 36, 37, 38, 39, 40]) * kHz)
make_g_plots(10, 'pairs', vary_g=True, g_list=-np.array([36, 38, 40]) * kHz)
